# 04 — Vector DB Indexing với ChromaDB

**Vai trò:** Data Engineer · **Task:** S3-DE-02 (Yêu cầu 9.2)

Notebook này khám phá `ChromaVectorStore` (S2-DE-03, S3-DE-01) — bước **Store** và **Retrieve** trong luồng `Load → Chunk → Embed → Store → Retrieve`. Ta lưu chunk + embedding thật vào ChromaDB (cả hai chế độ **in-memory** và **persistent**), truy vấn bằng `similarity_search()`, và xác minh trực tiếp Property 6/7/8 cùng `get_collection_stats()`/`delete_collection()`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker
from src.embeddings.embedding_model import OllamaEmbeddingModel
from src.embeddings.vector_store import ChromaVectorStore
from src.models import ChunkStrategy

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. Chuẩn bị: Load → Chunk → Embed

Tái sử dụng tài liệu mẫu từ [`01_document_loading.ipynb`](01_document_loading.ipynb) (tạo lại nếu thiếu — Yêu cầu 9.1), chia chunk, rồi tạo embedding thật qua `embed_batch()` (đảm bảo `embed_batch(texts)[i] == embed_text(texts[i])` — Property 5).

In [2]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
sample_path = RAW_DIR / "sample_rag_overview.txt"

if not sample_path.exists():
    sample_path.write_text(
        "Retrieval-Augmented Generation (RAG) la kien truc ket hop retrieval va "
        "generation. He thong tim cac doan van ban lien quan tu kho du lieu rieng "
        "truoc khi yeu cau LLM sinh cau tra loi, giup giam hien tuong ao giac va "
        "bam sat nguon tai lieu thuc te. Quy trinh RAG gom indexing va querying.",
        encoding="utf-8",
    )

document = DocumentLoader().load(str(sample_path))
chunker = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=180, chunk_overlap=30)
chunks = chunker.chunk(document)

embedder = OllamaEmbeddingModel(model_name="nomic-embed-text")
vectors = embedder.embed_batch([c.content for c in chunks])

assert len(vectors) == len(chunks)
assert vectors[0] == embedder.embed_text(chunks[0].content)  # Property 5
print(f"document.doc_id = {document.doc_id}")
print(f"So chunk        = {len(chunks)}")
print(f"Chieu embedding = {embedder.dimension}")

document.doc_id = 4c69f82b6454088f
So chunk        = 8
Chieu embedding = 768


## 2. `add()` — lưu chunks + vectors vào ChromaDB (in-memory)

`ChromaVectorStore` ở chế độ **in-memory** (`persist_dir=None`) — phù hợp để thực nghiệm nhanh trong notebook mà không tạo file trên đĩa (Yêu cầu 4.5, 4.6). `get_collection_stats()` cho biết số chunk hiện có trong collection.

In [3]:
memory_store = ChromaVectorStore(collection_name="notebook_vector_indexing", persist_dir=None)
ok = memory_store.add(chunks, vectors)
assert ok is True

stats = memory_store.get_collection_stats()
print(f"add() thanh cong: {ok}")
print(f"get_collection_stats(): {stats}")
assert stats["num_chunks"] == len(chunks)
assert stats["in_memory"] is True

add() thanh cong: True
get_collection_stats(): {'collection_name': 'notebook_vector_indexing', 'num_chunks': 8, 'persist_dir': None, 'in_memory': True}


## 3. `similarity_search()` — xác minh Property 6, 7, 8

Truy vấn bằng vector embedding của một câu hỏi mẫu, lấy về tối đa `k` kết quả. Theo design.md Phần 3:
- **Property 6**: `len(ket qua) <= k`
- **Property 7**: kết quả sắp xếp **giảm dần** theo `score`
- **Property 8**: mỗi `score ∈ [0.0, 1.0]`

(Validates Yêu cầu 4.2, 4.3, 4.4)

In [4]:
K = 3
query_vector = embedder.embed_text("RAG giam hien tuong ao giac nhu the nao?")
results = memory_store.similarity_search(query_vector, k=K)

print(f"similarity_search(k={K}) tra ve {len(results)} ket qua:")
for sc in results:
    preview = sc.chunk.content.strip().replace("\n", " ")[:80]
    print(f"  #{sc.rank} score={sc.score:.3f}  doc_id={sc.chunk.doc_id}  [{sc.chunk.start_index}:{sc.chunk.end_index}]  {preview!r}...")

scores = [sc.score for sc in results]
assert len(results) <= K, "Property 6: khong vuot qua k"
assert scores == sorted(scores, reverse=True), "Property 7: phai sap xep giam dan"
assert all(0.0 <= s <= 1.0 for s in scores), "Property 8: score phai thuoc [0.0, 1.0]"
assert all(sc.chunk.doc_id == document.doc_id for sc in results), "moi ket qua phai den tu tai lieu da index"
print("\nProperty 6, 7, 8 OK.")

similarity_search(k=3) tra ve 3 ket qua:
  #1 score=0.868  doc_id=4c69f82b6454088f  [355:534]  'nguon tai lieu thuc te va giam hien tuong "ao giac" (hallucination).  Quy trinh '...
  #2 score=0.842  doc_id=4c69f82b6454088f  [177:355]  'da hoc trong qua trinh huan luyen, mot he thong RAG se tim kiem cac doan van ban'...
  #3 score=0.822  doc_id=4c69f82b6454088f  [673:853]  'Du an nay trien khai toan bo quy trinh do bang cac thanh phan chay hoan toan cuc'...

Property 6, 7, 8 OK.


## 4. Persistent mode — lưu trữ lâu dài trên đĩa

Khi truyền `persist_dir`, `ChromaVectorStore` dùng `chromadb.PersistentClient` — dữ liệu tồn tại giữa các lần chạy (đây là chế độ dashboard dùng để lưu collection thật, xem `app/components/pipeline_factory.py`). Notebook dùng một thư mục **riêng** (`data/vector_db/notebook_demo`) để không ảnh hưởng tới collection chính của dashboard, và dọn dẹp ngay sau khi minh hoạ xong.

In [5]:
PERSIST_DIR = PROJECT_ROOT / "data" / "vector_db" / "notebook_demo"
persistent_store = ChromaVectorStore(collection_name="notebook_persistent_demo", persist_dir=str(PERSIST_DIR))
persistent_store.add(chunks, vectors)

stats = persistent_store.get_collection_stats()
print(f"get_collection_stats() (persistent): {stats}")
assert stats["in_memory"] is False
assert PERSIST_DIR.exists(), "PersistentClient phai tao thu muc luu tru tren dia"
print(f"Thu muc luu tru ton tai tren dia: {PERSIST_DIR.exists()}")

get_collection_stats() (persistent): {'collection_name': 'notebook_persistent_demo', 'num_chunks': 8, 'persist_dir': 'D:\\lh222k\\AI-Research-Assistant-with-RAG\\data\\vector_db\\notebook_demo', 'in_memory': False}
Thu muc luu tru ton tai tren dia: True


## 5. `delete_collection()` — dọn dẹp collection

Xóa collection vừa tạo để không để lại dữ liệu thừa — `delete_collection()` trả về `True` khi xóa thành công, `False` nếu collection không tồn tại (Yêu cầu 4.7).

In [6]:
deleted = persistent_store.delete_collection("notebook_persistent_demo")
print(f"delete_collection('notebook_persistent_demo') -> {deleted}")
assert deleted is True

deleted_again = persistent_store.delete_collection("khong_ton_tai_collection")
print(f"delete_collection('khong_ton_tai_collection') -> {deleted_again}  (collection khong ton tai)")
assert deleted_again is False

memory_store.delete_collection(memory_store.collection_name)
print(f"Da don dep collection in-memory '{memory_store.collection_name}'.")

delete_collection('notebook_persistent_demo') -> True
delete_collection('khong_ton_tai_collection') -> False  (collection khong ton tai)
Da don dep collection in-memory 'notebook_vector_indexing'.


## 6. Tổng kết

- `ChromaVectorStore.add()` lưu thành công cặp `(chunk, vector)` thật — mỗi `metadata` giữ `doc_id`/`start_index`/`end_index` để `similarity_search()` tái tạo lại đúng `Chunk` gốc.
- `similarity_search()` thoả mãn **Property 6/7/8**: không vượt quá `k`, sắp xếp giảm dần theo `score`, và `score ∈ [0.0, 1.0]` — nền tảng để `RAGPipeline.query()` luôn nhận được context hợp lệ.
- `ChromaVectorStore` hỗ trợ song song **in-memory** (notebook, thực nghiệm nhanh) và **persistent** (dashboard, lưu trữ lâu dài) qua cùng một interface (Yêu cầu 4.5, 4.6).
- `get_collection_stats()`/`delete_collection()` giúp theo dõi và dọn dẹp collection — hữu ích khi thực nghiệm nhiều cấu hình chunking/embedding khác nhau trên cùng một kho lưu trữ.